In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import altair as alt
import seaborn as sns
from IPython.display import display
import numpy as np

In [ ]:
os.getcwd()

In [ ]:
NodeType = {
  "INVALID_NODE": 0,
  "METADATA_NODE": 1,
  "MEM_LOAD_NODE": 2,
  "MEM_STORE_NODE": 3,
  "COMP_NODE": 4,
  "COMM_SEND_NODE": 5,
  "COMM_RECV_NODE": 6,
  "COMM_COLL_NODE": 7,
}
node_types = list(NodeType.keys())

In [ ]:
def get_timings_df(df: pd.DataFrame) -> pd.DataFrame:
    df_issues = df.query("action == 'issue'").drop(columns="action")
    df_callbacks = df.query("action == 'callback'").drop(columns="action")
    return df_issues.merge(
        df_callbacks,
        on=["sys_id", "node_id", "node_name", "node_type"],
        suffixes=("_issue", "_callback"),
    ).assign(elapsed_time=lambda d: d["tick_callback"] - d["tick_issue"])

def plot_elapsed_times(df: pd.DataFrame, sys_id: int = 0, max_height: int =600) -> alt.Chart:
    df = df.query(f"sys_id == {sys_id}")
    unique_nodes = df["node_name"].nunique()
    chart_height = unique_nodes * 20
    return alt.Chart(df).mark_bar().encode(
        x=alt.X('elapsed_time:Q', title='Elapsed Time'),
        y=alt.Y('node_name:N', sort=alt.SortField(field='tick_issue', order='ascending'), title='Node Name'),
        tooltip=['node_name', 'elapsed_time', 'tick_issue']
    ).properties(
        width=600,
        height=min(chart_height, max_height),
        title='Elapsed Time by Node Name'
    ).configure_axis(
        labelFontSize=10
    ).interactive()

def get_overlapped_blocks(df: pd.DataFrame) -> dict[str, list[int]]:
    """df should be filtered by sys_id and node_type."""
    df_sorted = df.sort_values("tick_issue")
    blocks = {
        "start": [],
        "end": [],
    }
    prev_start, prev_end = 0, 0
    for _, row in df_sorted.iterrows():
        if row["tick_issue"] <= prev_end:
            # overlap
            # merge the two blocks with proper start and end times
            # update the prev variables
            # do not append the block until we know it is not overlapping with any other subsequent block
            prev_end = max(prev_end, row["tick_callback"])
        else:
            # there is no overlap, append the block and update the prev variables
            blocks["start"].append(prev_start)
            blocks["end"].append(prev_end)
            prev_start = row["tick_issue"]
            prev_end = row["tick_callback"]
            
    blocks["start"].append(prev_start)
    blocks["end"].append(prev_end)
    return blocks

def plot_overlapped_blocks(df: pd.DataFrame) -> alt.Chart:
    chart = alt.Chart(df).mark_bar().encode(
        x=alt.X('start:Q', title='Timestamp'),
        x2='end:Q',
        y=alt.Y('node_type:N', title='Node Type'),
        color='node_type:N',  # Different color for each node_type
    ).configure_axis(
        grid=False,  # Remove the grid lines
        ticks=False
    ).properties(
        height=200,
        width=800,
        title='Duration of Blocks by Node Type'
    )
    
    return chart.interactive()

def plot_one_npu(df, npu, plot_blocks=True, plot_times=True):
    print("npu: ", npu)
    df_0 = df.query(f"sys_id == {npu}")
    df_0_comp = pd.DataFrame.from_dict(get_overlapped_blocks(df_0.query("node_type == 4"))).assign(
        node_type="COMPUTATION"
    )
    df_0_comm = pd.DataFrame.from_dict(get_overlapped_blocks(df_0.query("node_type in (5, 6, 7)"))).assign(
        node_type="COMMUNICATION"
    )
    df_0_blocks = pd.concat([df_0_comp, df_0_comm])
    if plot_blocks:
        display(plot_overlapped_blocks(df_0_blocks))
    if plot_times:
        display(plot_elapsed_times(df, max_height=600))

def plot_all_npus(df):
    for npu in range(64):
        df_0 = df.query(f"sys_id == {npu}")
        plot_one_npu(df, npu, plot_blocks=True, plot_times=False)

def plot_roofline(df, beta=5e10, pi=1e15):
    chart = alt.Chart(df).mark_circle().encode(
        x=alt.X('operational_intensity:Q'),
        y=alt.Y('perf:Q'),
        size=alt.value(100),
        #tooltip=['node_id', 'num_ops', 'tensor_size', 'perf', 'operational_intensity', 'elapsed_time']  # Show all columns
    ).properties(
        title='Performance vs Operational Intensity',
        width=600,
        height=400
    ).interactive()
    display(chart)

def plot_roofline(df, beta=5e10, pi=1e15):
    # Compute intersection point
    I_c = pi / beta
    P_c = pi

    # Compute x-axis domain so that I_c is at 2/3 of the plot
    log_Ic = np.log10(I_c)
    log_xmin = log_Ic - (log_Ic - 0) * 1.5  # shift left to make I_c at 2/3
    log_xmax = log_Ic + (log_Ic - log_xmin) / 2
    x_vals = np.logspace(log_xmin, log_xmax, 100)

    # Create dataframes for the roofline model lines
    roofline_data = pd.DataFrame({
        'operational_intensity': x_vals,
        'beta_line': beta * x_vals,
        'pi_line': [pi] * len(x_vals)
    })

    # Base scatter plot
    chart = alt.Chart(df).mark_circle().encode(
        x=alt.X('operational_intensity:Q', scale=alt.Scale(type='log'), title='Operational Intensity (FLOPs/byte)'),
        y=alt.Y('perf:Q', scale=alt.Scale(type='log'), title='Performance (FLOPs/sec)'),
        size=alt.value(100),
        tooltip=list(df.columns)
    ).properties(
        title='Roofline Model: Performance vs Operational Intensity',
        width=600,
        height=400
    )

    # Bandwidth line (sloped)
    beta_line = alt.Chart(roofline_data).mark_line(color='red').encode(
        x='operational_intensity:Q',
        y='beta_line:Q'
    )

    # Peak performance line (horizontal)
    pi_line = alt.Chart(roofline_data).mark_line(color='green').encode(
        x='operational_intensity:Q',
        y='pi_line:Q'
    )

    # Intersection point marker
    intersect_point = pd.DataFrame({
        'operational_intensity': [I_c],
        'perf': [P_c]
    })
    intersection = alt.Chart(intersect_point).mark_point(color='black', shape='cross', size=200).encode(
        x='operational_intensity:Q',
        y='perf:Q'
    )

    final_chart = (chart + beta_line + pi_line).interactive()
    display(final_chart)


# GPT 3 1300M 2D Torus

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/1_8_2_4_0_trace.csv")
df = get_timings_df(df)
plot_one_npu(df, 0)

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/4_2_2_4_0_trace.csv")
df = get_timings_df(df)
plot_one_npu(df, 0)
#plot_all_npus(df)

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/2_1_32_1_0_trace.csv")
df = get_timings_df(df)
#plot_one_npu(df, 0)
plot_all_npus(df)

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/4_1_16_1_0_trace.csv")
df = get_timings_df(df)
#plot_one_npu(df, 0)
plot_all_npus(df)

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/4_8_2_1_0_trace.csv")
df = get_timings_df(df)
plot_one_npu(df, 0)
#plot_all_npus(df)

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/1_8_2_4_0_roofline.csv")
plot_roofline(df.loc[:, ["perf", "operational_intensity"]].drop_duplicates())

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/2_1_32_1_0_roofline.csv")
plot_roofline(df.loc[:, ["perf", "operational_intensity"]].drop_duplicates())

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/4_1_16_1_0_roofline.csv")
plot_roofline(df.loc[:, ["perf", "operational_intensity"]].drop_duplicates())

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/4_2_2_4_0_roofline.csv")
plot_roofline(df.loc[:, ["perf", "operational_intensity"]].drop_duplicates())

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/4_8_2_1_0_roofline.csv")
plot_roofline(df.loc[:, ["perf", "operational_intensity"]].drop_duplicates())

In [ ]:
df = pd.read_csv("output/GPT_3_1300M/2D_Torus/16_1_1_4_0_roofline.csv")
plot_roofline(df.loc[:, ["perf", "operational_intensity"]].drop_duplicates())

In [ ]:
for arch in ["2D_Torus", "3D_Torus", "DGX1", "DGX_H100", "Dragonfly", "FullyConnected", "Ring", "Switch"]:
    df = pd.read_csv("output/GPT_3_1300M/3D_Torus/1_8_2_4_0_roofline.csv")
    print(f"{arch} 1_8_2_4_0")
    plot_roofline(df.loc[:, ["perf", "operational_intensity"]].drop_duplicates())


# Questions / next steps
* Do the computation and communication cycles match the ones obtained at the end of the output?
* Can we get the sizes of the **computation nodes** in flops or something? Check the [chatGPT response](https://chatgpt.com/c/67d88c68-3510-8007-bf07-d8712511c914)
* Can we get the sizes of the **communication nodes** in GB or something? Check the [chatGPT response](https://chatgpt.com/c/67d88c68-3510-8007-bf07-d8712511c914)
* Can we get some kind of direction on what is the bottleneck?
    * maybe a classification of the traces in something like:
        * memory constrained
        * compute constrained
        * comm constrained
 
    * separate in colors:
        * comp
        * comm
        * idle
        * then have an indicator across time, indicating if we have or not comp and comm. And see idle

* does astra sim have "resolution" on the memory accessess of data (L1, L2 cache etc)?
* there is a way to simulate an HBM with memory and latency. Does not include the size of the memory
* Since we are using STG, we get 2 types of nodes. There is a 3rd catgory of nodes: memory load / store. It would use this HBM model.

Jordi Ros
* can we modify incrementally with a delta the compute resources, memory etc, and get a sense of the "derivative" at each timestep.
Corti
* There is a technique called "design of experiments" in statistics that does this

# Next steps (from meeting)
* how complex would be to add memory to astrasim so it can be aware of OOM issues.
* The roofline is a bandwidth model. We can play incrementally with bandwidth and then see how the system would react to that, and see the gradient of modifying bandwidth.
* visualize for each gpu, plot a node as a point in the roofline model. Then we would have N points here and see if we are bottlenecked by comp, bandwidth or what.


* locate comm in the trace 

# Next steps 2025-03-27

2 directions
1. keep modelling memory
2. add NS3 or G2 for network
3. extend the STG to ZeRO family and ZeRO++
    * we will need to add memory nodes that are not present in STG
    * see how many more communication nodes shall we add to STG traces in order to 
4. increase speed of trace generation
5. the overall solver
6. metrics to quantify bottlenecks:
    * how idle is the computation, see proportions etc, shall we optimize the max, the average, etc